# 09 — Fastlog / Sparse Predicate Recording

**Purpose:** Show the sparse predicate recording workflow: `tl.record(save=...)` captures
only the ops that match a predicate, with far lower overhead than a full trace. Show what
a `Recording` looks like, how to inspect its event stream, how to cook it into a full
`Trace` via `to_trace()`, and edge-case behaviors: `halt`, module event visibility,
and `tl.fastlog.dry_run` for pre-flight label discovery.

**Surfaces covered:**
- [ ] `tl.record(model, x, save=...)` — sparse predicate recording entry point
- [ ] `Recording.__repr__` / `Recording.summary()` — overview of what was captured
- [ ] `Recording.n_ops` / `Recording.n_records` — counts
- [ ] `Recording.activation_payloads_by_raw_label` — tensor payloads
- [ ] `Recording.records` — `ActivationRecord` list; `.ctx.label`, `.ctx.func_name`
- [ ] `Recording.to_pandas()` — tabular view of recorded events
- [ ] `Recording.to_trace()` — cook sparse recording into a full `Trace`
- [ ] `Recording.recording_trace` — the underlying `RecordingTrace` event stream
- [ ] `RecordingTrace.to_pandas()` — full event table (ops + module enter/exit)
- [ ] `RecordingTrace.summary()` — counts by event kind
- [ ] `tl.fastlog.dry_run(model, x, keep_op=...)` — pre-flight label discovery
- [ ] `RecordingTrace.repredicate(other_keep_op=...)` — re-apply a predicate offline
- [ ] `halt=` kwarg — early-exit predicate on the recording
- [ ] `Recording.halted` / `Recording.halt_reason` — inspect halt result
- [ ] `tl.fastlog` module attributes (shim, types, helpers)
- [ ] `keep_op=` / `keep_module=` deprecated aliases vs `save=` canonical form

## 1. Setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens : {tl.__version__}")
print(f"torch     : {torch.__version__}")

## 2. The payoff: sparse vs full capture

A full `tl.trace` saves every tensor; `tl.record(save=pred)` saves only matching ops.
This section makes the contrast concrete.

In [ ]:
# _vocab: tl.record, tl.trace, Recording, Trace

torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

# Full trace — everything
full_trace = tl.trace(model, x)

# Sparse recording — only relu ops
rec_relu = tl.record(model, x, save=tl.func("relu"))

# Sparse recording — only linear ops
rec_linear = tl.record(model, x, save=tl.func("linear"))

print("Full trace layers     :", len(full_trace.layer_labels))
print("Recording(relu).n_ops :", rec_relu.n_ops)
print("Recording(linear).n_ops:", rec_linear.n_ops)
print()
print(
    "relu payloads saved  :",
    {k: v.shape for k, v in rec_relu.activation_payloads_by_raw_label.items()},
)
print(
    "linear payloads saved:",
    {k: v.shape for k, v in rec_linear.activation_payloads_by_raw_label.items()},
)

## 3. `Recording` repr and `summary()`

In [ ]:
# _vocab: Recording.__repr__, Recording.summary, Recording.n_ops, Recording.n_records

print("repr(rec_relu):", repr(rec_relu))
print()
print("rec_relu.summary():", rec_relu.summary())
print()
print("n_ops    :", rec_relu.n_ops, "  (ops that matched the save= predicate)")
print("n_records:", rec_relu.n_records, "  (events stored in records list)")

## 4. `Recording.to_pandas()` — tabular event view

In [ ]:
# _vocab: Recording.to_pandas

df_relu = rec_relu.to_pandas()
print("Recording(relu).to_pandas():")
print(df_relu.to_string())
print()
df_linear = rec_linear.to_pandas()
print("Recording(linear).to_pandas():")
print(df_linear.to_string())

## 5. `ActivationRecord` — the low-level record object

Each matched op appears in `rec.records` as an `ActivationRecord` with a `.ctx` context
field and payload attributes. The payload lives in `ram_payload` or `disk_payload`.

In [ ]:
# _vocab: Recording.records, ActivationRecord, RecordContext

# rec_relu has n_records=0 because n_records counts a different store;
# the actual tensor is in activation_payloads_by_raw_label
# The records list holds ActivationRecord objects for the matched ops:
print("len(rec_relu.records):", len(rec_relu.records))

# Note: n_records (summary count) != len(records) on this API -- see GAP callout below

# Access activation payloads directly
for raw_label, tensor in rec_relu.activation_payloads_by_raw_label.items():
    print(f"  payload['{raw_label}']: shape={tensor.shape}, dtype={tensor.dtype}")
print()

# Show other_keep_op field showing which predicate was applied
print("keep_op_repr :", rec_relu.keep_op_repr)
print("halted       :", rec_relu.halted)
print("halt_reason  :", rec_relu.halt_reason)

## 6. `Recording.to_trace()` — cook into a full Trace

`to_trace()` re-runs the captured event stream into a standard `Trace` with all the
normal accessor infrastructure. Unsaved payloads exist as metadata-only ops.

In [ ]:
# _vocab: Recording.to_trace

cooked = rec_relu.to_trace()
print("cooked type     :", type(cooked).__name__)
print("cooked repr     :", repr(cooked))
print("cooked layers   :", cooked.layer_labels)
print()
# All layers exist as metadata; only relu has a payload
print("relu activation via cooked trace:")
print(cooked["relu"].out)

## 7. `RecordingTrace` — the event stream under a Recording

`recording.recording_trace` exposes the full event stream (ops + module enter/exit events).
This is the low-level backbone that `to_trace()` reads.

In [ ]:
# _vocab: Recording.recording_trace, RecordingTrace.summary, RecordingTrace.to_pandas

rt = rec_relu.recording_trace
print("RecordingTrace type    :", type(rt).__name__)
print("RecordingTrace summary :", rt.summary())
print()
print("RecordingTrace.to_pandas() — full event table including module enter/exit:")
print(rt.to_pandas().to_string())

## 8. `tl.fastlog.dry_run` — pre-flight label discovery

`dry_run` runs the model but saves nothing — it returns only the `RecordingTrace`.
Use it to discover what raw labels exist before committing to a save predicate.

In [ ]:
# _vocab: tl.fastlog.dry_run, RecordingTrace

# dry_run: discover what labels are available (no payloads saved)
dry = tl.fastlog.dry_run(
    model,
    x,
    keep_op=lambda ctx: True,  # accept all so every op appears
)

print("dry_run type   :", type(dry).__name__)
print("dry_run summary:", dry.summary())
print()
print("Full event table (all ops + module events):")
print(dry.to_pandas().to_string())

## 9. `RecordingTrace.repredicate` — offline predicate replay

Already have a `RecordingTrace`? `repredicate(other_keep_op=...)` re-applies a new predicate
without re-running the model — useful for interactive label exploration.

In [ ]:
# _vocab: RecordingTrace.repredicate

rt_linear_only = dry.repredicate(other_keep_op=lambda ctx: ctx.layer_type == "linear")
print("repredicate (linear only) summary:", rt_linear_only.summary())
print()
print("repredicate decisions:")
print(rt_linear_only.to_pandas()[["kind", "op_type", "address", "shape"]].to_string())

## 10. `halt=` — early exit predicate

Returning `True` from `halt=` stops the forward pass immediately after the matching event.
Use it to capture only a prefix of the computation.

In [ ]:
# _vocab: halt=, Recording.halted, Recording.halt_reason

# Halt immediately after the first linear op; save it too
rec_halted = tl.record(
    model,
    x,
    save=tl.func("linear"),
    halt=lambda ctx: ctx.layer_type == "linear",  # stop at first linear
)

print("halted     :", rec_halted.halted)
print("halt_reason:", rec_halted.halt_reason)
print("n_ops (saved before halt):", rec_halted.n_ops)
print()
print("payloads captured before halt:")
for k, v in rec_halted.activation_payloads_by_raw_label.items():
    print(f"  {k}: {v.shape}")

## 11. `tl.fastlog` module — shim attributes and types

In [ ]:
# _vocab: tl.fastlog (module namespace)

public_fastlog = [a for a in dir(tl.fastlog) if not a.startswith("_")]
print("tl.fastlog public attributes:")
for name in sorted(public_fastlog):
    print(f"  {name}")
print()
print("tl.fastlog.record is tl.record:", tl.fastlog.record is tl.record)

## 12. `save=` vs deprecated `keep_op=` / `keep_module=` aliases

In [ ]:
# The canonical form is save=; keep_op= and keep_module= are deprecated aliases
# Both work; keep_op= emits a DeprecationWarning

import warnings

# Canonical form
rec_canonical = tl.record(model, x, save=tl.func("relu"))
print("canonical save=: keep_op_repr =", rec_canonical.keep_op_repr)
print()

# Legacy keep_op= alias (note: triggers DeprecationWarning)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    rec_legacy = tl.record(model, x, keep_op=tl.func("relu"))
    dep_warnings = [
        str(warning.message) for warning in w if issubclass(warning.category, DeprecationWarning)
    ]

print(f"keep_op= deprecation warnings ({len(dep_warnings)}):")
for dw in dep_warnings:
    print(f"  {dw[:120]}")
print()
print(
    "Both produce equivalent payloads:",
    set(rec_canonical.activation_payloads_by_raw_label)
    == set(rec_legacy.activation_payloads_by_raw_label),
)

## ⚠️ GAPs / ergonomic smells

- **`Recording.n_records` (in `summary()`) != `len(rec.records)`**: `n_records` reports 1
  when one op is saved, but `len(rec.records)` is also 1 — the count IS consistent after
  inspection, but the field name `n_records` in `summary()` vs the zero in attribute access
  initially looks contradictory. Both count activation records; the confusion is that
  `n_records=0` in the stored attribute path while `summary()` prints `n_records=1`.
  Likely a counting bug between the two code paths.
- **`tl.fastlog.dry_run` does not accept `save=`** — only `keep_op=` / `keep_module=`.
  Inconsistent with `tl.record(save=...)` canonical spelling. Must use the deprecated API
  for dry runs, which is backwards.
- **`RecordingTrace.repredicate` does not accept `save=`** — same issue; only `other_keep_op=`.
  All three call sites (record, dry_run, repredicate) use different kwarg names for the
  same concept.
- **`Recording.__repr__` is the full event dump** — very verbose (full RecordContext chain).
  A summary repr like `Recording(n_ops=1, n_records=1, save='tl.func(relu)')` would be far
  more readable for interactive work.
- **`tl.record` requires `save=` (or `keep_op=`) to be set** — calling `tl.record(model, x)`
  with no predicate raises `RecordingConfigError: fastlog requires a predicate or a true
  default capture spec`. Not immediately obvious; compare to `tl.trace` which just works.
  No clear path to "record everything" without using `tl.trace` instead.
- **Module events visible in `RecordingTrace.to_pandas()` but not in `Recording.to_pandas()`** —
  `recording_trace.to_pandas()` shows `module_enter`/`module_exit` rows; `recording.to_pandas()`
  shows only the matched op rows. This is intentional but should be documented more visibly.